In [41]:
# =============================================================================
# Conditional GAN (cGAN) for 128x128 images - Kaggle notebook (TensorFlow/Keras)
# =============================================================================

import os
import glob
import random
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt

from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.layers import (
    Conv2D,
    Conv2DTranspose,
    Dense,
    Flatten,
    Reshape,
    Dropout,
    Embedding,
    BatchNormalization,
    LeakyReLU,
)
from scipy.linalg import sqrtm
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
import time


In [42]:
# -----------------------------------------------------------------------------
# Spectral normalization (pure TensorFlow - Keras 3 Compatible)
# -----------------------------------------------------------------------------
@tf.keras.utils.register_keras_serializable(package="cgan")
class SpectralNormalization(tf.keras.layers.Wrapper):
    """Spectral norm on Conv2D or Dense weights (Miyato et al., 2018)."""

    def __init__(self, layer, power_iterations=1, **kwargs):
        super().__init__(layer, **kwargs)
        if power_iterations <= 0:
            raise ValueError("power_iterations must be > 0")
        self.power_iterations = power_iterations

    def build(self, input_shape):
        super().build(input_shape)
        
        # Keras 3 safety: convert shape to tuple instead of relying on TensorShape methods
        input_shape = tuple(input_shape)
        self.input_spec = tf.keras.layers.InputSpec(shape=(None,) + input_shape[1:])

        if hasattr(self.layer, "kernel"):
            self.w = self.layer.kernel
        elif hasattr(self.layer, "embeddings"):
            self.w = self.layer.embeddings
        else:
            raise AttributeError(
                f"{type(self.layer).__name__} has no 'kernel' or 'embeddings'"
            )

        # Keras 3 safety: use tuple() instead of .as_list()
        self.w_shape = tuple(self.w.shape)
        
        self.u = self.add_weight(
            shape=(1, self.w_shape[-1]),
            initializer=tf.initializers.TruncatedNormal(stddev=0.02),
            trainable=False,
            name="sn_u",
            dtype=self.w.dtype,
        )

    def call(self, inputs, training=None):
        if training is None:
            training = True
        if training:
            self.normalize_weights()
        return self.layer(inputs)

    def compute_output_shape(self, input_shape):
        # Keras 3 safety: directly return the layer's output shape
        return self.layer.compute_output_shape(input_shape)

    def normalize_weights(self):
        w = tf.reshape(self.w, [-1, self.w_shape[-1]])
        u = self.u
        with tf.name_scope("spectral_normalize"):
            for _ in range(self.power_iterations):
                v = tf.math.l2_normalize(tf.matmul(u, w, transpose_b=True))
                u = tf.math.l2_normalize(tf.matmul(v, w))
                u = tf.stop_gradient(u)
                v = tf.stop_gradient(v)
            sigma = tf.matmul(tf.matmul(v, w), u, transpose_b=True)
            self.u.assign(tf.cast(u, self.u.dtype))
            self.w.assign(
                tf.cast(tf.reshape(self.w / sigma, self.w_shape), self.w.dtype)
            )

    def get_config(self):
        base = super().get_config()
        return {**base, "power_iterations": self.power_iterations}

In [43]:
# --- Hyperparameters ---
LATENT_VECTOR_DIM = 128
BATCH_SIZE = 32
N_EPOCHS = 100
IMG_SIZE = 128  # Scaled up to 128x128

LABEL_SMOOTH = 0.1
GRAD_CLIP_NORM = 1.0
PER_CLASS = 100

LR = 2e-4
BETA1 = 0.5

# KID Parameters
KID_NUM_SAMPLES = 1024
KID_EPOCHS = 5
KID_INCEPTION_BATCH = 32

MODEL_SAVE_DIR = "/kaggle/working/"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [44]:
# =============================================================================
# Dataset: Optimized OpenCV loader for .tif histopathology images
# =============================================================================
import os
import glob
import random
import cv2
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt

DATA_DIR = os.environ.get(
    "CGAN_DATA_DIR",
    "/kaggle/input/datasets/imrankhan77/nct-crc-he-100k/NCT-CRC-HE-100K",
)

VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}")

class_names = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d)) and d.upper() != "BACK"
)
N_CLASSES = len(class_names)
class_to_label = {name: i for i, name in enumerate(class_names)}

file_paths, labels = [], []
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    all_files = glob.glob(os.path.join(cls_path, "*"))
    all_files = [fp for fp in all_files if os.path.splitext(fp)[1].lower() in VALID_EXT]
    random.shuffle(all_files)
    # Using the subset PER_CLASS limit defined in Cell 3 for faster epochs
    selected = all_files[:] 
    file_paths.extend(selected)
    labels.extend([class_to_label[cls]] * len(selected))

print(f"Total images: {len(file_paths)}")
file_paths = tf.constant(file_paths)
labels = tf.constant(labels, dtype=tf.int32)

def fast_tiff_load(path_bytes):
    """Loads and resizes TIFF images using OpenCV for maximum speed."""
    path = path_bytes.decode("utf-8")
    
    # OpenCV reads images in BGR format
    img = cv2.imread(path)
    if img is None:
        # Fallback safeguard in case of a corrupted file
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        
    # Convert BGR to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize (INTER_AREA is best for downsampling)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    
    # Normalize to [-1, 1] for Generator Tanh
    arr = np.asarray(img, dtype=np.float32)
    return (arr / 127.5) - 1.0

def load_image(path, label):
    # Wrap the OpenCV function back into the TensorFlow graph
    img = tf.numpy_function(fast_tiff_load, [path], tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label

dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
dataset = dataset.shuffle(buffer_size=len(file_paths), reshuffle_each_iteration=True)
dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

print("Dataset ready.")


Total images: 89434
Dataset ready.


In [45]:
# =============================================================================
# Generator: noise z + class label -> 128 x 128 x 3 image in [-1, 1]
# =============================================================================
class ConditionalGenerator(Model):
    """cGAN generator: (z, label) -> 128x128x3."""

    def __init__(
        self,
        n_classes,
        latent_dim,
        embed_dim=128,
        base_channels=512,
        img_channels=3,
        name="ConditionalGenerator",
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        self.n_classes = n_classes
        self.latent_dim = latent_dim
        self.embed_dim = embed_dim
        self.base_channels = base_channels

        self.label_embedding = Embedding(n_classes, embed_dim, name="g_label_embedding")
        self.fc = Dense(4 * 4 * base_channels, name="g_fc")
        self.reshape = Reshape((4, 4, base_channels), name="g_reshape")

        # Stack of Conv2DTranspose blocks: 4 -> 8 -> 16 -> 32 -> 64 -> 128
        self.g_tconv_8 = Conv2DTranspose(512, 4, strides=2, padding="same", use_bias=False, name="g_tconv_8")
        self.g_bn_8 = BatchNormalization(name="g_bn_8")

        self.g_tconv_16 = Conv2DTranspose(256, 4, strides=2, padding="same", use_bias=False, name="g_tconv_16")
        self.g_bn_16 = BatchNormalization(name="g_bn_16")

        self.g_tconv_32 = Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False, name="g_tconv_32")
        self.g_bn_32 = BatchNormalization(name="g_bn_32")

        self.g_tconv_64 = Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False, name="g_tconv_64")
        self.g_bn_64 = BatchNormalization(name="g_bn_64")
        
        self.g_tconv_128 = Conv2DTranspose(32, 4, strides=2, padding="same", use_bias=False, name="g_tconv_128")
        self.g_bn_128 = BatchNormalization(name="g_bn_128")

        self.g_out = Conv2D(
            img_channels,
            7,
            padding="same",
            activation="tanh",
            name="g_out_conv",
        )

    def call(self, inputs, training=False):
        z, label = inputs
        le = self.label_embedding(label)
        le = tf.squeeze(le, axis=1)
        x = tf.concat([z, le], axis=-1)
        x = self.fc(x)
        x = self.reshape(x)

        x = tf.nn.relu(self.g_bn_8(self.g_tconv_8(x), training=training))
        x = tf.nn.relu(self.g_bn_16(self.g_tconv_16(x), training=training))
        x = tf.nn.relu(self.g_bn_32(self.g_tconv_32(x), training=training))
        x = tf.nn.relu(self.g_bn_64(self.g_tconv_64(x), training=training))
        x = tf.nn.relu(self.g_bn_128(self.g_tconv_128(x), training=training))

        return self.g_out(x)

In [46]:
# =============================================================================
# Discriminator: (image, label) -> probability in (0, 1)
# =============================================================================
class ConditionalDiscriminator(Model):
    """cGAN discriminator: (image, label) -> fake/real score."""

    def __init__(
        self,
        n_classes,
        img_size=128,
        embed_dim=64,
        name="ConditionalDiscriminator",
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        self.n_classes = n_classes
        self.img_size = img_size

        self.label_embedding = Embedding(n_classes, embed_dim, name="d_label_embedding")
        # Project to a single channel (128x128x1) to save millions of parameters
        self.label_dense = Dense(img_size * img_size * 1, name="d_label_proj")
        self.label_reshape = Reshape((img_size, img_size, 1), name="d_label_reshape")

        self.leaky = LeakyReLU(0.2)

        def sn_conv(f, name_prefix):
            return SpectralNormalization(
                Conv2D(f, 4, strides=2, padding="same", name=f"{name_prefix}_conv"),
                name=f"{name_prefix}_sn",
            )

        # Sequence: 128 -> 64 -> 32 -> 16 -> 8 -> 4
        self.d_conv_64 = sn_conv(64, "d_64")
        self.d_conv_32 = sn_conv(128, "d_32")
        self.d_conv_16 = sn_conv(256, "d_16")
        self.d_conv_8 = sn_conv(512, "d_8")
        self.d_conv_4 = sn_conv(512, "d_4") 

        self.flatten = Flatten(name="d_flatten")
        self.drop = Dropout(0.3, name="d_dropout")
        self.d_dense = SpectralNormalization(
            Dense(1, activation="sigmoid", name="d_logit_dense"),
            name="d_dense_sn",
        )

    def call(self, inputs, training=False):
        img, label = inputs
        le = self.label_embedding(label)
        le = tf.squeeze(le, axis=1)
        le = self.label_dense(le)
        le = self.label_reshape(le)
        
        # Concatenate 3 RGB channels with 1 Label channel -> 4 channels total
        x = tf.concat([img, le], axis=-1)

        x = self.leaky(self.d_conv_64(x))
        x = self.leaky(self.d_conv_32(x))
        x = self.leaky(self.d_conv_16(x))
        x = self.leaky(self.d_conv_8(x))
        x = self.leaky(self.d_conv_4(x))

        x = self.flatten(x)
        x = self.drop(x, training=training)
        return self.d_dense(x)

In [47]:
# Build models (eager shape pass for Keras 3 / TF2)
G = ConditionalGenerator(n_classes=N_CLASSES, latent_dim=LATENT_VECTOR_DIM)
_ = G([tf.zeros((1, LATENT_VECTOR_DIM)), tf.zeros((1, 1), dtype=tf.int32)])

D = ConditionalDiscriminator(n_classes=N_CLASSES, img_size=IMG_SIZE)
_ = D([tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)), tf.zeros((1, 1), dtype=tf.int32)])

G.summary()
D.summary()

Model: "ConditionalGenerator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ g_label_embedding (Embedding)   │ (1, 1, 128)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_fc (Dense)                    │ (1, 8192)              │     2,105,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_reshape (Reshape)             │ (1, 4, 4, 512)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_tconv_8 (Conv2DTranspose)     │ (1, 8, 8, 512)         │     4,194,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_bn_8 (BatchNormalization)     │ (1, 8, 8, 512)         │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_tconv_16 (Conv2DTranspose)    │ (1, 16, 16, 256)       │     2,097,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_bn_16 (BatchNormalization)    │ (1, 16, 16, 256)       │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_tconv_32 (Conv2DTranspose)    │ (1, 32, 32, 128)       │       524,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_bn_32 (BatchNormalization)    │ (1, 32, 32, 128)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_tconv_64 (Conv2DTranspose)    │ (1, 64, 64, 64)        │       131,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_bn_64 (BatchNormalization)    │ (1, 64, 64, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_tconv_128 (Conv2DTranspose)   │ (1, 128, 128, 32)      │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_bn_128 (BatchNormalization)   │ (1, 128, 128, 32)      │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ g_out_conv (Conv2D)             │ (1, 128, 128, 3)       │         4,707 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,094,627 (34.69 MB)

 Trainable params: 9,092,643 (34.69 MB)

 Non-trainable params: 1,984 (7.75 KB)

Model: "ConditionalDiscriminator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ d_label_embedding (Embedding)   │ (1, 1, 64)             │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_label_proj (Dense)            │ (1, 16384)             │     1,064,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_label_reshape (Reshape)       │ (1, 128, 128, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_64_sn (SpectralNormalization) │ (1, 64, 64, 64)        │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_32_sn (SpectralNormalization) │ (1, 32, 32, 128)       │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_16_sn (SpectralNormalization) │ (1, 16, 16, 256)       │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_8_sn (SpectralNormalization)  │ (1, 8, 8, 512)         │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_4_sn (SpectralNormalization)  │ (1, 4, 4, 512)         │     4,195,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_flatten (Flatten)             │ (1, 8192)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_dropout (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ d_dense_sn                      │ (1, 1)                 │         8,194 │
│ (SpectralNormalization)         │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,027,522 (30.62 MB)

 Trainable params: 8,026,049 (30.62 MB)

 Non-trainable params: 1,473 (5.75 KB)

In [48]:
# =============================================================================
# Custom Training Loop
# =============================================================================
class GAN(Model):
    def __init__(
        self,
        generator,
        discriminator,
        latent_dim,
        label_smooth=0.1,
        grad_clip_norm=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.generator = generator
        self.discriminator = discriminator
        self.latent_dim = latent_dim
        self.label_smooth = label_smooth
        self.grad_clip_norm = grad_clip_norm

    def compile(self, opt_g, opt_d, loss_g, loss_d, **kwargs):
        super().compile(**kwargs)
        self.opt_g = opt_g
        self.opt_d = opt_d
        self.loss_g = loss_g
        self.loss_d = loss_d

    @tf.function
    def train_step(self, batch):
        images_real, labels = batch
        labels = tf.expand_dims(labels, axis=-1)
        bs = tf.shape(images_real)[0]
        ls = self.label_smooth

        # --- 1) Discriminator ---
        z = tf.random.normal(shape=(bs, self.latent_dim))
        images_fake = self.generator([z, labels], training=True)

        with tf.GradientTape() as d_tape:
            yhat_real = self.discriminator([images_real, labels], training=True)
            yhat_fake = self.discriminator([images_fake, labels], training=True)
            
            y_real = tf.ones_like(yhat_real) * (1.0 - ls)
            y_fake = tf.zeros_like(yhat_fake) + ls
            loss_d = 0.5 * (
                self.loss_d(y_real, yhat_real) + self.loss_d(y_fake, yhat_fake)
            )

        d_grads = d_tape.gradient(loss_d, self.discriminator.trainable_variables)
        if self.grad_clip_norm and self.grad_clip_norm > 0:
            d_grads, _ = tf.clip_by_global_norm(d_grads, self.grad_clip_norm)
        self.opt_d.apply_gradients(zip(d_grads, self.discriminator.trainable_variables))

        # --- 2) Generator ---
        z = tf.random.normal(shape=(bs, self.latent_dim))
        with tf.GradientTape() as g_tape:
            images_fake = self.generator([z, labels], training=True)
            yhat_fake = self.discriminator([images_fake, labels], training=False)
            
            y_gen = tf.ones_like(yhat_fake) * (1.0 - ls)
            loss_g = self.loss_g(y_gen, yhat_fake)

        g_grads = g_tape.gradient(loss_g, self.generator.trainable_variables)
        if self.grad_clip_norm and self.grad_clip_norm > 0:
            g_grads, _ = tf.clip_by_global_norm(g_grads, self.grad_clip_norm)
        self.opt_g.apply_gradients(zip(g_grads, self.generator.trainable_variables))

        return {"loss_d": loss_d, "loss_g": loss_g}

In [49]:
# =============================================================================
# Kernel Inception Distance (KID)
# =============================================================================
def polynomial_kernel(X, Y, degree=3, gamma=None, coef0=1):
    if gamma is None:
        gamma = 1.0 / X.shape[1]
    K = np.dot(X, Y.T)
    K *= gamma
    K += coef0
    K **= degree
    return K

def kid_from_features(real_features, fake_features):
    m = real_features.shape[0]
    n = fake_features.shape[0]

    K_XX = polynomial_kernel(real_features, real_features)
    K_YY = polynomial_kernel(fake_features, fake_features)
    K_XY = polynomial_kernel(real_features, fake_features)

    np.fill_diagonal(K_XX, 0)
    np.fill_diagonal(K_YY, 0)

    term_XX = np.sum(K_XX) / (m * (m - 1))
    term_YY = np.sum(K_YY) / (n * (n - 1))
    term_XY = 2.0 * np.sum(K_XY) / (m * n)

    return float((term_XX + term_YY - term_XY))

class KIDCallback(tf.keras.callbacks.Callback):
    def __init__(self, dataset, latent_dim, num_samples=250, every=5, inception_batch=32, save_dir="./"):
        super().__init__()
        self.dataset = dataset
        self.latent_dim = latent_dim
        self.num_samples = num_samples
        self.every = every
        self.inception_batch = inception_batch
        self.save_dir = save_dir
        self._inception = None
        self.kid_history = []
        self.best_kid = float('inf')
        self.best_epoch = 0

    def _get_inception(self):
        if self._inception is None:
            self._inception = InceptionV3(
                include_top=False, pooling="avg", weights="imagenet", input_shape=(299, 299, 3)
            )
            self._inception.trainable = False
        return self._inception

    def _features(self, images_nhwc):
        inc = self._get_inception()
        out = []
        n = images_nhwc.shape[0]
        for i in range(0, n, self.inception_batch):
            batch = images_nhwc[i : i + self.inception_batch]
            x = tf.convert_to_tensor(batch, dtype=tf.float32)
            x = tf.image.resize(x, [299, 299])
            x = (x + 1.0) / 2.0 * 255.0
            x = preprocess_input(x)
            feat = inc(x, training=False)
            out.append(feat.numpy())
        return np.concatenate(out, axis=0)
    
    def save_best_model(self, epoch, kid):
        if kid < self.best_kid:
            self.best_kid = kid
            self.best_epoch = epoch
            
            gen_path = os.path.join(self.save_dir, f"generator_best_kid_{kid:.4f}_epoch_{epoch}.weights.h5")
            self.model.generator.save_weights(gen_path)
            
            disc_path = os.path.join(self.save_dir, f"discriminator_best_kid_{kid:.4f}_epoch_{epoch}.weights.h5")
            self.model.discriminator.save_weights(disc_path)
            
            print(f"KID : {kid:.4f} at epoch {epoch}")

    def on_epoch_end(self, epoch, logs=None):
        if self.every <= 0 or (epoch + 1) % self.every != 0:
            return

        print(f"\nComputing KID at epoch {epoch + 1} ({self.num_samples} pairs)...")
        imgs_list, labs_list = [], []
        n = 0
        for x, y in self.dataset.repeat():
            imgs_list.append(x.numpy())
            labs_list.append(y.numpy())
            n += x.shape[0]
            if n >= self.num_samples: break

        imgs = np.concatenate(imgs_list, axis=0)[: self.num_samples]
        labs = np.concatenate(labs_list, axis=0)[: self.num_samples]

        z = tf.random.normal((self.num_samples, self.latent_dim))
        labs_t = tf.expand_dims(tf.constant(labs, dtype=tf.int32), -1)
        fake = self.model.generator([z, labs_t], training=False).numpy()

        f_real = self._features(imgs.astype(np.float32))
        f_fake = self._features(fake.astype(np.float32))
        
        kid = kid_from_features(f_real, f_fake)

        print(f"KID  ≈ {kid:.4f}  (lower is better)")
        self.kid_history.append((epoch + 1, kid))
        self.save_best_model(epoch + 1, kid)
        if logs is not None: logs["kid"] = kid

In [50]:
# =============================================================================
# Training Execution & Callbacks
# =============================================================================
bce = BinaryCrossentropy(from_logits=False)

gan = GAN(
    generator=G,
    discriminator=D,
    latent_dim=LATENT_VECTOR_DIM,
    label_smooth=LABEL_SMOOTH,
    grad_clip_norm=GRAD_CLIP_NORM,
)
gan.compile(
    opt_g=Adam(LR, beta_1=BETA1),
    opt_d=Adam(LR, beta_1=BETA1),
    loss_g=bce,
    loss_d=bce,
)

class GenerateSamplesCallback(tf.keras.callbacks.Callback):
    def __init__(self, n_classes, latent_dim, class_names, grid_size=3, every=10):
        super().__init__()
        self.n_classes = n_classes
        self.latent_dim = latent_dim
        self.class_names = class_names
        self.grid_size = grid_size
        self.every = every
        self.fixed_z = tf.random.normal((grid_size * grid_size, latent_dim))
        labels = np.arange(grid_size * grid_size) % n_classes
        self.fixed_labels = tf.expand_dims(labels, axis=-1)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every != 0:
            return
        print(f"\nSamples at epoch {epoch + 1}")
        images = self.model.generator([self.fixed_z, self.fixed_labels], training=False).numpy()
        images = np.clip((images + 1) / 2.0, 0, 1)
        fig, ax = plt.subplots(self.grid_size, self.grid_size, figsize=(8, 8))
        idx = 0
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                ax[i, j].imshow(images[idx])
                lid = int(self.fixed_labels[idx])
                ax[i, j].set_title(self.class_names[lid], fontsize=8)
                ax[i, j].axis("off")
                idx += 1
        plt.tight_layout()
        plt.show()

class SpecificEpochCheckpoint(tf.keras.callbacks.Callback):
    """Saves the generator and discriminator at explicitly defined epochs."""
    def __init__(self, target_epochs, save_dir="./"):
        super().__init__()
        self.target_epochs = target_epochs
        self.save_dir = save_dir

    def on_epoch_end(self, epoch, logs=None):
        current_epoch = epoch + 1
        if current_epoch in self.target_epochs:
            gen_path = os.path.join(
                self.save_dir, 
                f"generator_epoch_{current_epoch}.weights.h5"
            )
            disc_path = os.path.join(
                self.save_dir, 
                f"discriminator_epoch_{current_epoch}.weights.h5"
            )
            
            self.model.generator.save_weights(gen_path)
            self.model.discriminator.save_weights(disc_path)
            
            print(f" Saved model weights for epoch {current_epoch}")

sample_callback = GenerateSamplesCallback(
    n_classes=N_CLASSES,
    latent_dim=LATENT_VECTOR_DIM,
    class_names=class_names,
    grid_size=3,
    every=5,
)

kid_callback = KIDCallback(
    dataset=dataset,
    latent_dim=LATENT_VECTOR_DIM,
    num_samples=KID_NUM_SAMPLES,
    every=KID_EPOCHS,
    inception_batch=KID_INCEPTION_BATCH,
    save_dir=MODEL_SAVE_DIR,
)

epoch_saver_callback = SpecificEpochCheckpoint(
    target_epochs=[60, 70, 80, 90, 100],
    save_dir=MODEL_SAVE_DIR
)
start_time = time.time()
history = gan.fit(
    dataset,
    epochs=N_EPOCHS,
    callbacks=[epoch_saver_callback],
)
end_time = time.time()
training_duration = end_time - start_time
hours, rem = divmod(training_duration, 3600)
minutes, seconds = divmod(rem, 60)

print(f"Total Training Time: {int(hours):02d}h {int(minutes):02d}m {int(seconds):02d}s")

Epoch 1/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 362s 122ms/step - loss_d: 0.5309 - loss_g: 1.9775
Epoch 2/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6219 - loss_g: 1.5237
Epoch 3/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6348 - loss_g: 1.3316
Epoch 4/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6451 - loss_g: 1.1573
Epoch 5/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6557 - loss_g: 1.0513
Epoch 6/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6599 - loss_g: 1.0063
Epoch 7/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6624 - loss_g: 0.9553
Epoch 8/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6679 - loss_g: 0.9195
Epoch 9/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6696 - loss_g: 0.8687
Epoch 10/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: 0.6757 - loss_g: 0.7989
Epoch 11/100
2794/2794 ━━━━━━━━━━━━━━━━━━━━ 343s 123ms/step - loss_d: